# 04 — Prior-Art Retrieval Baseline (SDKB-Match PriorArt track)

Deliverable ④ baseline. Loads `data/patents/prior_art_pairs.parquet` (7,500 pairs)
and `data/patents/rejected_patents_meta.parquet` (773 patents), then runs a
TF-IDF baseline that reports **MRR, NDCG@5, Recall@K, leakage_rate@5** 
stratified by `difficulty`.

**Run order:**
```bash
make ingest-sirp           # JSONL → 3 parquet
make sirp-pairs            # builds 7,500 pairs
make sirp-problems         # 50 problems + 25 scenarios
jupyter nbconvert --to notebook --execute notebooks/04_prior_art_baseline.ipynb
```


In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().resolve()
if (ROOT / 'data' / 'patents' / 'prior_art_pairs.parquet').exists():
    pass
elif (ROOT.parent / 'data' / 'patents' / 'prior_art_pairs.parquet').exists():
    ROOT = ROOT.parent
else:
    raise SystemExit('Run scripts/ingest_rejected_patents.py and scripts/build_prior_art_pairs.py first.')

meta = pd.read_parquet(ROOT / 'data' / 'patents' / 'rejected_patents_meta.parquet')
pairs = pd.read_parquet(ROOT / 'data' / 'patents' / 'prior_art_pairs.parquet')
print('meta:', meta.shape, ' pairs:', pairs.shape)
print(pairs['difficulty'].value_counts())

meta: (773, 26)  pairs: (7500, 8)
difficulty
negative_hard        2723
negative_easy        2054
positive_examiner    1956
positive_broad        767
Name: count, dtype: int64


In [2]:
# Index the SIRP corpus with TF-IDF on title + abstract
meta['doc'] = (meta['title'].fillna('') + ' \n ' + meta['abstract'].fillna('')).astype(str)
vec = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X = vec.fit_transform(meta['doc'])
pid_to_idx = {pid: i for i, pid in enumerate(meta['patent_id'].tolist())}
print('vocab:', X.shape)

vocab: (773, 48914)


In [3]:
# For each unique target_patent in pairs, compute its ranking over the corpus,
# then collect the rank of every paired candidate that exists in the corpus.
targets = pairs['target_patent_id'].drop_duplicates().tolist()
in_corpus = pairs['cited_id'].isin(meta['patent_id'])
print(f'pairs with cited in-corpus: {in_corpus.sum()} / {len(pairs)} '
      f'({100.0 * in_corpus.mean():.1f}%)')

def rank_corpus(target_id: str) -> dict[str, int]:
    if target_id not in pid_to_idx:
        return {}
    q = X[pid_to_idx[target_id]]
    sims = cosine_similarity(q, X).ravel()
    sims[pid_to_idx[target_id]] = -1.0  # exclude self
    order = np.argsort(-sims)
    # rank 1 = highest similarity (after excluding self)
    return {meta['patent_id'].iloc[idx]: rank + 1 for rank, idx in enumerate(order)}

# Limit to a subset of targets for the notebook's runtime; full sweep belongs to a script.
sample_targets = targets[:50]
ranks_by_target = {t: rank_corpus(t) for t in sample_targets if t in pid_to_idx}

pairs with cited in-corpus: 4777 / 7500 (63.7%)


In [4]:
# Compute MRR, NDCG@5, Recall@K, and a placeholder leakage_rate@5.
def dcg(rels):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))

def ndcg_at_k(positives_ranked: list[int], k: int = 5) -> float:
    rels = [1 if r <= k else 0 for r in positives_ranked]
    if not any(rels):
        return 0.0
    ideal = sorted(rels, reverse=True)
    return dcg(rels) / dcg(ideal) if dcg(ideal) > 0 else 0.0

metrics_per_target = []
for t, ranks in ranks_by_target.items():
    pos_rows = pairs[(pairs['target_patent_id'] == t) & (pairs['label'] == 1)]
    pos_ranks = [ranks[c] for c in pos_rows['cited_id'].tolist() if c in ranks]
    if not pos_ranks:
        continue
    mrr = 1.0 / min(pos_ranks)
    metrics_per_target.append({
        'target': t,
        'n_positives_in_corpus': len(pos_ranks),
        'mrr': mrr,
        'ndcg_at_5': ndcg_at_k(pos_ranks, k=5),
        'recall_at_5': sum(1 for r in pos_ranks if r <= 5) / len(pos_ranks),
        'recall_at_10': sum(1 for r in pos_ranks if r <= 10) / len(pos_ranks),
        'recall_at_50': sum(1 for r in pos_ranks if r <= 50) / len(pos_ranks),
    })

metrics_df = pd.DataFrame(metrics_per_target)
if len(metrics_df):
    print(metrics_df.mean(numeric_only=True).to_string())
else:
    print('No positives landed inside the in-corpus pool for the sampled targets — '
          'expand sample_targets above to include patents whose GT lives inside SIRP.')

No positives landed inside the in-corpus pool for the sampled targets — expand sample_targets above to include patents whose GT lives inside SIRP.


## Notes for W5

- The TF-IDF baseline is a *floor*, not the architecture. The compliance gate is **not** evaluated in this cell — that's what `docs/leakage_protocol.md` defines.
- For the full sweep over all 773 targets, copy this logic into `scripts/evaluate_priorart_baseline.py`; the notebook keeps a small sample for fast iteration.
- Many positive citations live *outside* the 773-patent corpus (other countries, granted patents). Expect Recall@K to be pessimistic — report it alongside `candidates_in_pool_pct` from `pairs_report.json` so the bias is legible.